In [ ]:
import pandas as pd
from PIL import Image
import os
from openslide import OpenSlide
import glob
Image.MAX_IMAGE_PIXELS = None  # Disable PIL's limit on image size
SLIDE_DIRS = ["/mnt/data/Datasets/HE_data/CPTAC_UCEC/*.svs",
              "/mnt/data/Datasets/HE_data/ENDO-AID/data/*.tif",
              "/mnt/data/Datasets/HE_data/Labaj_UCEC/SVS/05_2024/*.svs",
              "/mnt/data/Datasets/HE_data/OCELOT/ocelot2023_v1.0.1/images/train/tissue/*.jpg",
              "/mnt/data/Datasets/HE_data/Rosella_HE_mice/Orig_tissue_tiff/*.tiff",
              "/mnt/data/Datasets/HE_data/Rosella_HE_mice/raw/*.ndpi",
              "/mnt/data/Datasets/HE_data/TCGA_UCEC/*.svs",
              "/mnt/data/Tmp/jmerta/tncb/Images/*.png"
              ]

DATASETS = [
    "CPTAC_UCEC",
    "ENDO-AID",
    "Labaj_UCEC",
    "OCELOT",
    "Rosella_HE_mice_Orig_tissue_tiff",
    "Rosella_HE_mice_raw",
    "TCGA_UCEC",
    "tncb"
]

dataset_list = []
slide_names = []
extensions = []
properties = []
backends = []

for slide_pattern, dataset_name in zip(SLIDE_DIRS, DATASETS):
    print(f"Examining pattern: {slide_pattern}")
    files = glob.glob(slide_pattern)
    if len(files) == 0:
        print(f"  Warning: no files matched pattern: {slide_pattern}")
        continue

    for file_path in files:
        s = os.path.basename(file_path)
        print(f"  - {s}")
        ext = os.path.splitext(s)[1]
        extensions.append(ext)
        dataset_list.append(dataset_name)
        slide_names.append(s)
        try:
            slide = OpenSlide(file_path)
            props_string = "\n".join([f"{k}: {v}" for k, v in slide.properties.items()])
            properties.append(props_string)
            backends.append("openslide")
        except Exception as e:
            try:
                slide = Image.open(file_path)
                props_string = "\n".join([f"{k}: {v}" for k, v in slide.info.items()])
                properties.append(props_string)
                backends.append("PIL")
            except Exception as e2:
                properties.append(f"ERROR: {str(e2)}")
                backends.append("ERROR")


df = pd.DataFrame({
    'Dataset': dataset_list,
    'Slide': slide_names,
    'Extension': extensions,
    'Properties': properties,
    'Backend': backends
})

# write output
df.to_csv("slide_properties.csv", index=False)


Examining pattern: /mnt/data/Datasets/HE_data/CPTAC_UCEC/*.svs
  - C3L-02557-22.svs
